In [3]:
%cd ..

c:\Users\HP\OneDrive - University of Moratuwa\Desktop\E-Vision-Projects\DB_SQL_GEN


In [4]:
import httpx
from loguru import logger
from src.config import settings

In [5]:
class ChatBotAuthClient:
    def __init__(self):
        self.auth_url = getattr(
            settings,
            "chatbot_auth_url"
        )
        self.api_key = getattr(settings, "chatbot_api_key", "123")

    async def authenticate_user(self, phone_no: str) -> dict:
        headers = {
            "Content-Type": "application/json",
            "CHatBot-Key": self.api_key
        }

        payload = {
            "Phoneno": phone_no
        }

        try:
            async with httpx.AsyncClient(timeout=30) as client:
                response = await client.post(
                    self.auth_url,
                    json=payload,
                    headers=headers
                )

            response.raise_for_status()
            return response.json()

        except Exception as e:
            logger.error(f"AuthenticateUser API failed: {e}")
            raise


chatbot_auth_client = ChatBotAuthClient()

In [6]:
from typing import Any


def get_value(data: dict, *keys, default=None):
    for key in keys:
        if isinstance(data, dict) and key in data:
            return data[key]
    return default


def normalize_list(value: Any) -> list:
    if value is None:
        return []

    if isinstance(value, list):
        return value

    if isinstance(value, str):
        return [x.strip() for x in value.split(",") if x.strip()]

    return []


def extract_allowed_rep_codes(auth_data: dict) -> list[str]:
    """
    Extract allowed RepCodes for SQL filter:
    FIND_IN_SET(shn.Code, @AllowedNodes) > 0

    For Rep response:
      use allowedNodes[].Code

    For Customer response:
      use rep.Code or rep.NodeCode
    """

    if not auth_data:
        return []

    success = get_value(auth_data, "success", "Success", default=True)
    if success is False:
        return []

    user_type = get_value(auth_data, "userType", "UserType", default="")

    allowed_codes = []

    # Rep response: allowedNodes contains accessible hierarchy nodes
    allowed_nodes = get_value(auth_data, "allowedNodes", "AllowedNodes", default=[])

    for node in normalize_list(allowed_nodes):
        if isinstance(node, dict):
            code = get_value(node, "Code", "code", "NodeCode", "nodeCode")
            if code:
                allowed_codes.append(str(code).strip())
        elif isinstance(node, str):
            allowed_codes.append(node.strip())

    # Some APIs may return direct allowed rep code arrays
    for key in ["allowedRepCodes", "AllowedRepCodes", "repCodes", "RepCodes"]:
        for code in normalize_list(auth_data.get(key)):
            if isinstance(code, str):
                allowed_codes.append(code.strip())
            elif isinstance(code, dict):
                value = get_value(code, "Code", "code", "RepCode", "repCode")
                if value:
                    allowed_codes.append(str(value).strip())

    # Customer response: assigned rep object
    rep = get_value(auth_data, "rep", "Rep", default=None)
    if isinstance(rep, dict):
        rep_code = get_value(
            rep,
            "Code",
            "code",
            "NodeCode",
            "nodeCode",
            "RepCode",
            "repCode"
        )
        if rep_code:
            allowed_codes.append(str(rep_code).strip())

    # Remove duplicates
    return sorted(set(x for x in allowed_codes if x))


def extract_user_role(auth_data: dict) -> str:
    user_type = get_value(auth_data, "userType", "UserType", default=None)

    if user_type:
        return str(user_type).lower()

    if get_value(auth_data, "rep", "Rep"):
        return "rep"

    if get_value(auth_data, "customer", "Customer"):
        return "customer"

    return "unknown"


def extract_user_context(auth_data: dict) -> dict:
    return {
        "user_type": extract_user_role(auth_data),
        "allowed_rep_codes": extract_allowed_rep_codes(auth_data),
        "raw_auth": auth_data
    }

In [7]:
auth_data = await chatbot_auth_client.authenticate_user(phone_no="011255332")

2026-05-24 09:39:54.265 | ERROR    | __main__:authenticate_user:31 - AuthenticateUser API failed: Client error '404 Not Found' for url 'http://144.76.225.16:26521/ChatBot/AuthenticateUser'
For more information check: https://developer.mozilla.org/en-US/docs/Web/HTTP/Status/404


HTTPStatusError: Client error '404 Not Found' for url 'http://144.76.225.16:26521/ChatBot/AuthenticateUser'
For more information check: https://developer.mozilla.org/en-US/docs/Web/HTTP/Status/404

In [6]:
auth_data

{'success': True,
 'userType': 'Customer',
 'customer': {'Id': 1100035,
  'Code': 'AMBREP003-000035',
  'Name': 'Alawaththa Stores',
  'DisplayName': None,
  'PhoneNo': '0779538759',
  'MobileNo': '0770500992',
  'EMail': None},
 'route': {'Id': 36, 'Code': 'RT028', 'Name': 'DEWATA BAGAHAGODA IMADUWA'},
 'rep': {'NodeId': 10,
  'NodeCode': 'AMBREP003',
  'NodeName': 'Ambalangoda Rep 03',
  'PdaImei': '773985381D5BDB16-AMBREP003',
  'UserId': 110,
  'UserCode': 'REP0022',
  'UserName': 'Ashen Tharuka-Ambalangoda',
  'MobileNo': None,
  'EMail': None}}

In [7]:
extract_user_context(auth_data)

{'user_type': 'customer',
 'allowed_rep_codes': ['AMBREP003'],
 'raw_auth': {'success': True,
  'userType': 'Customer',
  'customer': {'Id': 1100035,
   'Code': 'AMBREP003-000035',
   'Name': 'Alawaththa Stores',
   'DisplayName': None,
   'PhoneNo': '0779538759',
   'MobileNo': '0770500992',
   'EMail': None},
  'route': {'Id': 36, 'Code': 'RT028', 'Name': 'DEWATA BAGAHAGODA IMADUWA'},
  'rep': {'NodeId': 10,
   'NodeCode': 'AMBREP003',
   'NodeName': 'Ambalangoda Rep 03',
   'PdaImei': '773985381D5BDB16-AMBREP003',
   'UserId': 110,
   'UserCode': 'REP0022',
   'UserName': 'Ashen Tharuka-Ambalangoda',
   'MobileNo': None,
   'EMail': None}}}